In [ ]:
using Mangal
using DataFrames
using CSV
using Graphs
using PyCall
nx = pyimport("networkx")
np = pyimport("numpy")

function get_all_networks()

    println("Retrieving all Mangal networks...")

    number_of_networks = count(MangalNetwork)
    count_per_page = 100
    number_of_pages = convert(Int, ceil(number_of_networks / count_per_page))

    mangal_networks = []

    global cursor = 1
    for page in 1:number_of_pages

        println("\tProcessing page $page / $number_of_pages")

        global cursor
        networks_in_page = Mangal.networks("count" => count_per_page, "page" => page - 1)

        for current_network in networks_in_page
            push!(mangal_networks, current_network)
            cursor = cursor + 1
        end
    end

    println("Done. Total networks retrieved: $(length(mangal_networks))")

    return mangal_networks
end

function get_network_interactions(network::MangalNetwork)
    
    println("\tRetrieving all interactions for network id=$(network.id)...")

    # use pagination
    number_of_interactions = count(MangalInteraction, network)
    count_per_page = 100
    number_of_pages = convert(Int, ceil(number_of_interactions / count_per_page))

    mangal_interactions = []

    global cursor = 1
    for page in 1:number_of_pages
        println("\t\tProcessing page $page / $number_of_pages")

        global cursor
        interactions_in_page = interactions(network, "count" => count_per_page, "page" => page - 1)
        for current_interaction in interactions_in_page
            push!(mangal_interactions, current_interaction)
            cursor = cursor + 1
        end
    end

    println("\tTotal interactions retrieved: $(length(mangal_interactions))")

    return mangal_interactions

end

function build_adjacency_matrix(network_interactions, network_nodes)

    println("\tBuilding adjacency matrix...")

    tot_nodes = length(network_nodes)
    adj_matrix = zeros(Int, tot_nodes, tot_nodes)

    node_index = Dict{Int,Int}()
    for (idx, node) in enumerate(network_nodes)
        node_index[node.id] = idx
    end

    for interaction in network_interactions
        try
            from_idx = node_index[interaction.from.id]
            to_idx = node_index[interaction.to.id]
            adj_matrix[from_idx, to_idx] = 1
        catch e
            println("Warning: Interaction from id=$(interaction.from.id) to id=$(interaction.to.id) could not be mapped to nodes.")
        end
    end

    # Compute number of links
    num_links = sum(adj_matrix)

    println("\tAdjacency matrix built. Number of nodes: $tot_nodes, Number of links: $num_links")

    return adj_matrix
end

function get_node_names(network_nodes)

    node_names = String[]
    for node in network_nodes
        push!(node_names, node.name)
    end

    return node_names
end

function process_mangal_networks(mangal_networks; overwrite=false)

    println("### Processing Mangal networks ###")

    k = 0
    for n in mangal_networks

        # Check if file already exists
        output_file = "Data/networks/mangal_$(n.id).graphml"
        if isfile(output_file) && !overwrite
            println("File $output_file already exists. Skipping network id=$(n.id).")
            k += 1
            continue
        end

        println("Processing network $k / $(length(mangal_networks))")

        # Retrieve interactions
        network_interactions = get_network_interactions(n)

        # Retrieve nodes from interactions
        network_nodes = unique(vcat([interaction.from for interaction in network_interactions], [interaction.to for interaction in network_interactions]))

        # Build adjacency matrix
        adj_matrix = build_adjacency_matrix(network_interactions, network_nodes)

        # If adjacency matrix is not symmetric, symmetrize it (for undirected networks)
        adj_matrix = adj_matrix .| adj_matrix'

        # Get node names
        node_names = get_node_names(network_nodes)

        adj_matrix_numpy = np.array(adj_matrix)

        g = SimpleGraph(adj_matrix)

        G = nx.from_numpy_array(adj_matrix_numpy)

        if is_bipartite(g)

            m = bipartite_map(g)

            top_nodes = [node_names[i] for i in keys(m) if m[i] == 1]
            bottom_nodes = [node_names[i] for i in keys(m) if m[i] == 2]

            # Set node attributes for names and bipartite sets
            nx.set_node_attributes(G, Dict(i - 1 => node_names[i] for i in keys(m)), "name")
            nx.set_node_attributes(G, Dict(i - 1 => (m[i] == 1 ? "bottom" : "top") for i in keys(m)), "bipartite_set")

        end

        # Graph metadata
        G.graph["dataset"] = "mangal"
        G.graph["ID"] = string("mangal_", n.id)

        # Save the graph to a file
        nx.write_graphml(G, output_file)

        k += 1

    end

    println("### Processing complete ###")

end


"""
Classifies ecological interactions from a description string.
Uses a hierarchy: Pair co-occurrence -> Keyword Regex -> Default.
"""
function classify_interaction_type(description::String)
    # Normalize to lowercase once
    desc = lowercase(description)

    # --- Tier 1: Pair Co-occurrence Rules (Priority) ---
    # These override general keywords because they indicate specific relationships
    if occursin("flower", desc) && (occursin("insect", desc) || occursin("bee", desc))
        return "plant_pollinator"
    end

    if (occursin("fruit", desc) || occursin("seed", desc)) && (occursin("bird", desc) || occursin("frugivor", desc))
        return "seed_dispersal"
    end

    if occursin("ant", desc) && occursin("plant", desc)
        return "plant_ant"
    end

    if occursin("hummingbirds", desc)
        return "plant_pollinator"
    end

    # --- Tier 2: Pre-compiled Regex Patterns ---
    # Patterns use | for OR and [- \s]? to handle spaces or hyphens
    patterns = [
        # Predation / Food Webs
        r"food[-\s]?web|trophic|predation|diet|feeding|consumer|predator|prey" => "food_web",

        # Parasitism
        r"host[-\s]?parasite|parasitism|parasitoid" => "host_parasite",

        # Pollination
        r"pollinat|pollnator|flower[-\s]?visit" => "plant_pollinator",

        # Seed Dispersal
        r"seed[-\s]?dispersal|dispersal|frugivory" => "seed_dispersal",

        # Anemone - Fish
        r"anemone[-\s]?fish|anemon|clownfish|mutualism|symbiosis" => "anemone_fish",

        # Competition
        r"competition|competitive" => "competition"
    ]

    for (pattern, label) in patterns
        if occursin(pattern, desc)
            return label
        end
    end

    # --- Tier 3: Fallback ---
    return "other"
end

"""
Return (lon, lat) from either:
- a 2-vector [lon, lat]
- a vector of 2-vectors [[lon,lat], ...] defining a polygon ring (closed or not)

For polygons, returns the planar centroid (shoelace formula).
Falls back to mean of vertices if polygon area is ~0 (degenerate).
"""
function lonlat_or_centroid(x; atol_area=1e-14)
    # Case 1: single point
    if x isa AbstractVector && length(x) == 2 && !(first(x) isa AbstractVector)
        return (float(x[1]), float(x[2]))  # (lon, lat)
    end

    # Case 2: polygon ring as vector of points
    if x isa AbstractVector && !isempty(x) && first(x) isa AbstractVector && length(first(x)) == 2
        pts = x
        n = length(pts)
        n < 3 && error("Need at least 3 vertices for a polygon centroid.")

        # ensure ring closure for iteration
        lon1 = float(pts[1][1])
        lat1 = float(pts[1][2])
        lonN = float(pts[end][1])
        latN = float(pts[end][2])
        closed = (lon1 == lonN) && (lat1 == latN)

        # Shoelace centroid
        A2 = 0.0            # 2*Area
        Cx_num = 0.0
        Cy_num = 0.0

        # iterate over edges (i -> j)
        last = closed ? n - 1 : n
        for i in 1:last
            j = i + 1
            if j > n
                j = 1
            end
            xi = float(pts[i][1])
            yi = float(pts[i][2])
            xj = float(pts[j][1])
            yj = float(pts[j][2])

            cross = xi * yj - xj * yi
            A2 += cross
            Cx_num += (xi + xj) * cross
            Cy_num += (yi + yj) * cross
        end

        if abs(A2) <= atol_area
            # Degenerate polygon: return mean of vertices
            lons = [float(p[1]) for p in pts]
            lats = [float(p[2]) for p in pts]
            return (sum(lons) / length(lons), sum(lats) / length(lats))
        end

        lonc = Cx_num / (3 * A2)
        latc = Cy_num / (3 * A2)
        return (lonc, latc)
    end

    error("Unsupported input format. Expected [lon,lat] or [[lon,lat], ...].")
end


function construct_metadata_dataframe(mangal_networks)

    ids = []
    names = []
    datasets = []
    publics = []
    dates = []
    completes = []
    users = []
    descriptions = []
    lats = []
    lons = []

    println("Constructing metadata DataFrame from Mangal networks...")

    for n in mangal_networks
        push!(ids, n.id)
        push!(names, n.name)
        push!(datasets, n.dataset)
        push!(publics, n.public)
        push!(dates, n.date)
        push!(completes, n.complete)
        push!(users, n.user)
        push!(descriptions, n.description)

        # Extract geographic location if available
        geographic_location = n.position #Either lat/lon or a polygon

        if ! ismissing(geographic_location)
            clon, clat = lonlat_or_centroid(geographic_location)
            push!(lons, clon)
            push!(lats, clat)
        else
            push!(lons, missing)
            push!(lats, missing)
        end

    end

    println("Metadata extraction complete. Constructing DataFrame...")

    network_metadata = DataFrame(
        id=ids,
        name=names,
        dataset=datasets,
        public=publics,
        date=dates,
        complete=completes,
        user=users,
        description=descriptions,
        lon=lons,
        lat=lats
    )

    println("Metadata DataFrame constructed with $(nrow(network_metadata)) entries.")

    # Classify interaction types based on descriptions
    println("Classifying interaction types based on descriptions...")

    network_metadata.interaction_type = classify_interaction_type.(network_metadata.description)

    println("Interaction type classification complete. Distribution:")
    println(combine(groupby(network_metadata, :interaction_type), nrow => :count))

    # Save metadata to CSV
    CSV.write("Data/mangal/mangal_networks_metadata.csv", network_metadata)

    println("Metadata saved to Data/mangal/mangal_networks_metadata.csv")

    return network_metadata

end

In [ ]:
# mangal_networks = networks() # only first 100 networks

mangal_networks = get_all_networks()

In [ ]:
process_mangal_networks(mangal_networks, overwrite=false)

# Metadata

In [ ]:
mangal_networks = get_all_networks()

In [ ]:
network_metadata = construct_metadata_dataframe(mangal_networks)